In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
import os


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
    device_map="auto"
)


excel_path = "/content/multiple_categories_news_summarized.xlsx"
df = pd.read_excel(excel_path)


df.dropna(subset=['content', 'description'], inplace=True)


df = df[df['content'].str.strip() != '']
df = df[df['description'].str.strip() != '']

dataset = Dataset.from_pandas(df)


if len(dataset) < 2:
    raise ValueError("Dataset is too small to be split. Please check your source file and column names.")

dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]
print(f"Training samples: {len(train_dataset)}, Evaluation samples: {len(eval_dataset)}")


def preprocess_function(examples):

    inputs = []
    for content, description in zip(examples["content"], examples["description"]):

        prompt = f"Summarize the following article:\n\n{content}\n\nSummary:\n{description}{tokenizer.eos_token}"
        inputs.append(prompt)


    model_inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=1024)

    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs


train_dataset = train_dataset.map(
    preprocess_function, batched=True, remove_columns=train_dataset.column_names
)
eval_dataset = eval_dataset.map(
    preprocess_function, batched=True, remove_columns=eval_dataset.column_names
)


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./qwen_finetuned_summarization",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=10,
    report_to="none",
    fp16=False,
    bf16=True if device == "cuda" else False,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training for summarization...")
trainer.train()


model.save_pretrained("./qwen_finetuned_summarization")
tokenizer.save_pretrained("./qwen_finetuned_summarization")

print("Fine-tuning for summarization is complete!")

Using device: cuda
Training samples: 149, Evaluation samples: 17


Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Starting training for summarization...


Step,Training Loss,Validation Loss


Fine-tuning for summarization is complete!


In [ ]:

!zip -r qwen_finetuned_summarization.zip ./qwen_finetuned_summarization


from google.colab import files
files.download('qwen_finetuned_summarization.zip')


  adding: qwen_finetuned_summarization/ (stored 0%)
  adding: qwen_finetuned_summarization/vocab.json (deflated 61%)
  adding: qwen_finetuned_summarization/chat_template.jinja (deflated 71%)
  adding: qwen_finetuned_summarization/adapter_config.json (deflated 57%)
  adding: qwen_finetuned_summarization/added_tokens.json (deflated 67%)
  adding: qwen_finetuned_summarization/merges.txt (deflated 57%)
  adding: qwen_finetuned_summarization/tokenizer_config.json (deflated 89%)
  adding: qwen_finetuned_summarization/adapter_model.safetensors (deflated 8%)
  adding: qwen_finetuned_summarization/tokenizer.json (deflated 81%)
  adding: qwen_finetuned_summarization/special_tokens_map.json (deflated 69%)
  adding: qwen_finetuned_summarization/checkpoint-57/ (stored 0%)
  adding: qwen_finetuned_summarization/checkpoint-57/vocab.json (deflated 61%)
  adding: qwen_finetuned_summarization/checkpoint-57/optimizer.pt (deflated 8%)
  adding: qwen_finetuned_summarization/checkpoint-57/trainer_state.json

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

ADAPTER_PATH = "./qwen_finetuned_summarization"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto"
)


model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)



model.eval()

excel_path = "/content/multiple_categories_news_summarized.xlsx"
df = pd.read_excel(excel_path)
df.dropna(subset=['content'], inplace=True)
sample_text = df['content'].iloc[0] 

prompt = f"Summarize the following article:\n\n{sample_text}\n\nSummary:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

print("Generating summary...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        # max_new_tokens=128, 
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=True,      
        temperature=0.7,     
        top_p=0.9,           
    )


generated_summary = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print("\n--- Original Article ---\n")
print(sample_text)
print("\n--- Generated Summary ---\n")
print(generated_summary)

Using device: cuda
Generating summary...

--- Original Article ---

For decades, the government has grown increasingly reliant on the private sector to perform functions once handled by federal employees, a shift done ostensibly to control costs by having companies c… [+2980 chars]

--- Generated Summary ---

The US government is now spending more than $1 billion per year to train private contractors for jobs that
